In [ ]:
!pip install -q -U transformers peft accelerate datasets trl bitsandbytes

In [ ]:
# 1. 환경 설정 및 라이브러리 설치
# ----------------------------------------------------------------------
# -q (quiet) : 설치 과정을 간략하게 표시
# -U (upgrade) : 최신 버전으로 설치
#
# transformers: 허깅페이스 모델(Gemma 2) 로드 및 사용
# peft(Parameter Efficient Fine Tuning): LoRA, QLoRA 등 파라미터 효율적 파인튜닝 (PEFT)
# accelerate: PyTorch 모델 학습/추론을 쉽게 분산/가속
# datasets: CSV, JSON 등 데이터를 쉽게 로드하고 처리
# trl: SFTTrainer 등 LLM 파인튜닝을 위한 특수 도구로 작은 어댑터만 추가하여 훈련시간 단축
# bitsandbytes: 4비트 양자화(QLoRA)에 필수적인 라이브러리

import os
import pandas as pd
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,  # gemma-2-2b와 같은 모델 ID만 지정하면 알아서 해당 구조에 맞는 모델 클래스를 찾아 로드
    AutoTokenizer, # 모델 ID에 맞는 토크나이저 로드
    BitsAndBytesConfig,  # 파라미터들을 32비트(float32)가 아닌 4비트 양자화 설정하여 VRAM 사용량을 줄임
    TrainingArguments  # 모델 훈련에 필요한 모든 설정을 정의
)
from peft import (
    LoraConfig,   # LORa(Low Rank Adaptation)의 설정을 정의
    get_peft_model,  # 원본모델(4비트로 양자화된 AutoModel)과 LoraConfig를 입력받아,가중치는 고정되고 LoRa 어댑터가 적용된 PEFT모델 반환
    prepare_model_for_kbit_training  # 양자화(4비트, 8비트)된 모델을 훈련에 맞게 준비시켜주는 헬퍼(helper) 함수
)
from trl import SFTTrainer  # PEFT모델, 토크나이저, 데이터셋, TrainingArguments를 모두 전달받아 훈련을 실행 (ex. trainer.train())

In [ ]:
# 2. 허깅페이스 로그인 (Gemma 2 모델 접근)
# ----------------------------------------------------------------------
# Gemma 2 모델은 접근 권한이 필요합니다.
# 1. https://huggingface.co/google/gemma-2-2b-it 에서 접근 요청
# 2. https://huggingface.co/settings/tokens 에서 'WRITE' 권한 토큰 생성
# 3. 아래 실행 후 나오는 입력창에 생성한 토큰을 붙여넣기
from huggingface_hub import login
login()

In [ ]:
# 3. (실행용) 임시 데이터셋 생성
# 파일 경로를 'dataset/indata_kor2.csv'로 지정합니다.
# 'dataset' 폴더가 없을 경우를 대비해 os.makedirs로 폴더를 생성합니다.
#
# (참고) 실제 파일이 이미 Colab 환경의 'dataset' 폴더 안에 있다면
#        이 3번 셀은 실행하지 않고 건너뛰셔도 됩니다.

os.makedirs("dataset", exist_ok=True)

import os.path

# ========== [수정 1] 파일이 없을 때만 임시 데이터 생성 ==========
if not os.path.exists("dataset/indata_kor2.csv"):
    csv_data = """질문,답변
AI란 무엇인가요?,AI는 인공지능(Artificial Intelligence)의 약자로, 인간의 학습 능력, 추론 능력, 지각 능력 등을 컴퓨터 프로그램으로 실현한 기술입니다.
딥러닝과 머신러닝의 차이는?,머신러닝은 데이터에서 패턴을 학습하는 광범위한 분야이며, 딥러닝은 인공 신경망을 사용하여 머신러닝을 수행하는 한 분야입니다.
로라(LoRA) 파인튜닝은 왜 사용하나요?,LoRA는 'Low-Rank Adaptation'의 약자로, LLM의 모든 가중치를 업데이트하는 대신, 훨씬 적은 수의 어댑터 가중치만 추가하여 학습시킵니다.
Gemma 2 모델의 특징은 무엇인가요?,Gemma 2는 Google DeepMind에서 개발한 경량화된 고성능 언어 모델입니다. 2B와 9B, 27B 모델이 있으며, 적은 리소스로도 우수한 성능을 보여줍니다.
한국의 수도는 어디인가요?,한국(대한민국)의 수도는 서울입니다.
신경망이란?,신경망은 생물학적 신경계에서 영감을 받아 만든 수학적 모델로, 노드(뉴런)와 가중치로 구성되어 복잡한 패턴을 학습합니다.
트랜스포머 모델은?,트랜스포머는 자기주의(Self-Attention) 메커니즘을 기반으로 한 신경망 아키텍처로, 자연어 처리에서 혁신을 가져왔습니다.
파인튜닝이란?,파인튜닝은 사전학습된 모델을 특정 작업에 맞게 추가로 학습시키는 과정으로, 적은 데이터와 시간으로 높은 성능을 얻을 수 있습니다.
GPU와 CPU의 차이는?,GPU는 병렬 처리에 최적화되어 있어 머신러닝 학습에 유리하며, CPU는 순차 처리에 최적화되어 있습니다.
자연어 처리(NLP)란?,자연어 처리는 인간의 언어를 컴퓨터가 이해하고 처리할 수 있도록 하는 인공지능 분야입니다.
벡터화란 무엇인가요?,벡터화는 텍스트나 이미지 같은 데이터를 수치 벡터로 변환하는 과정으로, 머신러닝 모델이 처리할 수 있는 형태로 만듭니다.
"""

    # 'dataset/indata_kor2.csv' 파일로 저장
    with open("dataset/indata_kor2.csv", "w", encoding="utf-8") as f:
        f.write(csv_data)

    print("임시 dataset/indata_kor2.csv 파일 생성 완료.")
    print("--- [파일 내용 미리보기] ---")
    df = pd.read_csv("dataset/indata_kor2.csv")
    print(df)
    print("---------------------------")
else:
    print("기존 dataset/indata_kor2.csv 파일 사용")
    df = pd.read_csv("dataset/indata_kor2.csv")
    print(f"--- [파일 정보] ---")
    print(f"총 {len(df)}개 행 로드됨")
    print("---------------------------")

기존 dataset/indata_kor2.csv 파일 사용 (1000개 데이터)
--- [파일 정보] ---
총 1000개 행 로드됨
---------------------------


In [ ]:
# 4. 모델 및 토크나이저 로드 (QLoRA 적용)
# ----------------------------------------------------------------------
# QLoRA (Quantized LoRA): 4비트 양자화를 통해 모델을 극도로 압축하여
# Colab의 T4 GPU (16GB VRAM)에서도 2B 모델을 로드하고 학습할 수 있게 함.

model_id = "google/gemma-2-2b-it" # IT = Instruction Tuned (챗봇/Q&A에 적합)

# 4-1. BitsAndBytesConfig 설정 (4비트 양자화)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,  # 4비트로 로드
    bnb_4bit_use_double_quant=True,  # 2차 양자화로 메모리 추가 절약
    bnb_4bit_quant_type="nf4",  # NF4 (Normalized Float 4) 포맷 사용 (성능 저하 최소화)
    bnb_4bit_compute_dtype=torch.bfloat16  # 계산 시 bfloat16 사용 (Gemma의 기본 타입)
)

# 4-2. 토크나이저 로드
# Gemma 2 토크나이저는 다국어를 지원하며 한국어 성능이 우수합니다.
tokenizer = AutoTokenizer.from_pretrained(model_id)

# 4-3. 모델 로드 (양자화 설정 적용)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,  # 4비트 양자화 적용
    device_map="auto",  # 사용 가능한 GPU에 모델 레이어를 자동으로 분배
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/241M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

In [ ]:
# 5. 데이터셋 로드, 필터링 및 전처리
# ----------------------------------------------------------------------

# 5-1. Q&A 데이터를 Gemma 2의 채팅 템플릿으로 변환하는 함수
def format_dataset(example):
    messages = [
        {"role": "user", "content": example['질문']},
        {"role": "model", "content": example['답변']}
    ]
    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": formatted_text}

# 5-2. 데이터 유효성 검사 및 필터링 함수
# '질문' 또는 '답변' 열이 비어있거나(None) 공백만 있으면 제외합니다.
def check_data(example):
    is_question_valid = example['질문'] is not None and example['질문'].strip() != ""
    is_answer_valid = example['답변'] is not None and example['답변'].strip() != ""
    return is_question_valid and is_answer_valid

# 5-3. CSV 파일 로드 (경로 수정됨)
dataset = load_dataset("csv", data_files="dataset/indata_kor2.csv", split="train")
print(f"원본 데이터 {len(dataset)}개 로드 완료.")

# 5-4. 데이터 필터링 적용
filtered_dataset = dataset.filter(check_data)

# 필터링 결과 확인
original_count = len(dataset)
filtered_count = len(filtered_dataset)
print(f"데이터 필터링: 원본 {original_count}개 -> 유효 {filtered_count}개. ({original_count - filtered_count}개 제거됨)")

# 만약 유효한 데이터가 하나도 없다면 오류 발생
if filtered_count == 0:
    raise ValueError("!!! 치명적 오류: 유효한 데이터가 0개입니다. indata_kor2.csv 파일을 확인하세요.")
elif original_count > filtered_count:
    print("!!! 경고: '질문' 또는 '답변'이 비어있는 행이 제거되었습니다.")


# 5-5. 포맷팅 적용
# 'dataset'이 아닌 'filtered_dataset'을 사용
formatted_dataset = filtered_dataset.map(format_dataset)

# 5-6. 학습 데이터 최종 샘플 확인
# 여기서 'None'이 보이는지 꼭 확인하세요.
print("\n--- [학습에 실제 사용될 데이터 샘플] ---")
print(formatted_dataset[0]['text'])
print("------------------------------------------")


Generating train split: 0 examples [00:00, ? examples/s]

원본 데이터 1000개 로드 완료.


Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

데이터 필터링: 원본 1000개 -> 유효 1000개. (0개 제거됨)


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]


--- [학습에 실제 사용될 데이터 샘플] ---
<bos><start_of_turn>user
휴먼퓨처소프트의 CEO는 누구인가요?<end_of_turn>
<start_of_turn>model
현재 휴먼퓨처소프트의 대표이사(CEO)는 이지원 님입니다.<end_of_turn>

------------------------------------------


In [ ]:
# 6. LoRA 설정 (PEFT)
# ----------------------------------------------------------------------
# 모델의 어느 부분에 LoRA 어댑터를 적용할지 설정합니다.

# 6-1. (권장) 4비트 모델 학습을 위한 모델 준비
# 그래디언트 체크포인팅 등을 활성화하여 메모리를 더욱 절약
model = prepare_model_for_kbit_training(model)

# 6-2. LoRA Config 설정
lora_config = LoraConfig(
    r=16,  # LoRA 랭크 (높을수록 표현력은 좋지만 파라미터가 많아짐, 8 또는 16이 보편적)
    lora_alpha=32,  # 랭크의 2배 (alpha/r 비율로 스케일링)

    # Gemma 2 모델에서 LoRA를 적용할 레이어 타겟팅
    # (주의: 모델 아키텍처마다 다름, gemma 2는 아래 타겟이 효과적)
    # q_proj, k_proj, v_proj : 입력데이터를 Q(Query), K(Key), V(Value)로 투영하는 가중치 행렬로 어떤 부분에 집중, 정보추출할지 학습
    # o_proj : Attention 출력을 다시 시퀀스 출력 공간으로 투영하는 행렬. 결과를 효과적으로 통합하는 방법을 학습
    # gate_proj, up_proj, down_proj : Feed-Forward Network의 가중치 행렬로, 모델의 깊은 이해력과 추론 능력을 강화
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_dropout=0.1,  # 과적합 방지를 위한 드롭아웃
    bias="none",   # none: Bias 가중치는 학습하지 않고 고정, all: 모든 Bias 가중치 학습, lora_only: LoRa가 적용된 레이어의 Bias만 학습
    task_type="CAUSAL_LM"  # 현재 파인튜닝하려는 모델이 이전에 등장한 토큰들을 기반으로 다음 토큰을 예측하는 인과적언어모델임을 나타냄
)

# 6-3. 모델에 LoRA 어댑터 추가
model = get_peft_model(model, lora_config)

# 학습 가능한 파라미터 수 확인
model.print_trainable_parameters()

trainable params: 20,766,720 || all params: 2,635,108,608 || trainable%: 0.7881


In [ ]:
# 7. 학습(Training) 설정
# ----------------------------------------------------------------------
output_dir = "gemma2-2b-it-sft-lora-kor"  # 학습 결과가 저장될 폴더

training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,

    # 작은 데이터셋에서 50 에포크는 과적합 유발
    num_train_epochs=2,

    # 학습률 스케줄러 개선
    learning_rate=5e-4,
    lr_scheduler_type="linear",  # 학습률 스케줄러 (선형 감소)
    warmup_steps=1,
    warmup_ratio=0.05,

    # 로깅 및 저장 빈도
    logging_steps=10,
    save_strategy="steps",  # epoch 대신 steps로 변경 (cf. steps: 모델의 가중치를 한번 업데이트하는 단위)
    save_steps=100,
    save_total_limit=3,  # 최근 3개 체크포인트만 유지

    # 학습 안정성 개선
    gradient_checkpointing=True,  # 메모리 절약 + 안정성
    max_grad_norm=1.0,  # 그래디언트 클리핑 (기본값)
    weight_decay=0.01,  # L2 정규화로 과적합 방지

    # 평가 설정 조건부 변경
    eval_strategy="steps" if filtered_count > 3 else "no",  # 평가 데이터 있을 때만 평가
    eval_steps=100,
    load_best_model_at_end=True if filtered_count > 3 else False,  # 조건부
    metric_for_best_model="eval_loss" if filtered_count > 3 else None,  # 조건부
    greater_is_better=False,

    # bf16=True: Colab의 A100/T4 등 Ampere GPU에서 bf16 사용 (Gemma에 최적화)
    # fp16=False: bf16을 사용하므로 fp16은 끔
    bf16=True,
    fp16=False,
    optim="paged_adamw_8bit",  # QLoRA를 위한 8비트 페이징 옵티마이저 (메모리 절약)
    report_to="none",
    seed=42,
)

In [ ]:
# 8. SFTTrainer 설정 및 학습 시작
# 1000개 데이터를 8:2로 분할
if filtered_count >= 4:
    # 최소 3개의 학습 데이터를 보장하면서 분할
    test_ratio = min(0.15, (filtered_count - 3) / filtered_count)
    train_test_split = formatted_dataset.train_test_split(test_size=test_ratio, seed=42)
    train_dataset = train_test_split["train"]
    eval_dataset = train_test_split["test"]
else:
    # 데이터가 3개 이하면 전부 학습에 사용 (평가 데이터 없음)
    print("경고: 데이터가 적어 전체를 학습에 사용합니다.")
    train_dataset = formatted_dataset
    eval_dataset = formatted_dataset

print(f"학습 데이터: {len(train_dataset)}개")
print(f"평가 데이터: {len(eval_dataset)}개")


trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset if len(eval_dataset) > 0 else None,
    args=training_args,
)

print("\n--- [SFT (LoRA) 학습 시작] ---")
trainer.train()
print("--- [학습 완료] ---")

학습 데이터: 850개
평가 데이터: 150개


Adding EOS to train dataset:   0%|          | 0/850 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/850 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/850 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

...

Truncating eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.



--- [SFT (LoRA) 학습 시작] ---


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,0.071400,0.072199,0.077084,61071.000000,0.967643
200,0.070600,0.069420,0.073462,121690.000000,0.967738


--- [학습 완료] ---


In [ ]:
# 9. 학습된 모델(어댑터) 저장
# ----------------------------------------------------------------------
# 파인튜닝은 전체 모델을 저장하는 것이 아니라,
# 가볍게 추가된 'LoRA 어댑터'만 저장합니다.

adapter_path = "gemma2-lora-adapters-kor"
trainer.save_model(adapter_path)

print(f"\n학습된 LoRA 어댑터가 '{adapter_path}' 폴더에 저장되었습니다.")
print("이제 이 어댑터를 원본 Gemma 2 모델에 꽂아서 사용할 수 있습니다.")


학습된 LoRA 어댑터가 'gemma2-lora-adapters-kor' 폴더에 저장되었습니다.
이제 이 어댑터를 원본 Gemma 2 모델에 꽂아서 사용할 수 있습니다.


In [ ]:
# 10. 파인튜닝된 모델 테스트
# 저장된 어댑터를 로드하여 추론(Inference)을 수행합니다.

# model, trainer 존재 여부 확인 후 삭제 ==========
try:
    del model, trainer  # VRAM 확보
except NameError:
    # 이미 삭제됨 또는 정의되지 않음 (별도 셀에서 실행할 때)
    pass

torch.cuda.empty_cache()

from peft import PeftModel

print("✅ 모델 로드 중...")

# 필요한 변수 정의 (별도 셀에서 실행할 때)
if 'model_id' not in locals():
    model_id = "google/gemma-2-2b-it"

if 'bnb_config' not in locals():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )

if 'adapter_path' not in locals():
    adapter_path = "gemma2-lora-adapters-kor"

if 'tokenizer' not in locals():
    tokenizer = AutoTokenizer.from_pretrained(model_id)

# 10-1. 원본 4비트 모델 다시 로드
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# 10-2. 로드한 모델에 저장된 LoRA 어댑터 적용
model_with_adapters = PeftModel.from_pretrained(base_model, adapter_path)

print("✅ 모델 로드 완료!")
print("\n--- [파인튜닝된 모델 테스트] ---")

# ============================================================
# 질문 테스트 함수 (편리하게 재사용 가능)
# ============================================================
def test_question(question_text, description=""):
    """
    질문을 입력받아 모델의 답변을 생성하고 출력하는 함수

    Args:
        question_text: 질문 문자열
        description: 질문 설명 (옵션)
    """
    if description:
        print(f"\n--- [{description}] ---")

    messages = [
        {"role": "user", "content": question_text}
    ]

    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    outputs = model_with_adapters.generate(
        input_ids,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        top_p=0.9,
        repetition_penalty=1.1,
    )

    response = tokenizer.decode(outputs[0][input_ids.shape[-1]:], skip_special_tokens=True)

    print(f"질문: {question_text}")
    print(f"답변: {response}")

# ============================================================
# 테스트 질문들
# ============================================================

# 질문 1
test_question(
    "휴먼금융소프트는 언제 설립되었나요?",
    "테스트 1"
)

# 질문 2
test_question(
    "휴먼금융소프트의 주요 제품은 무엇인가요?",
    "테스트 2"
)

# 질문 3
test_question(
    "휴먼금융소프트에 대해 알려주세요.",
    "테스트 3"
)

✅ 모델 로드 중...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

✅ 모델 로드 완료!

--- [파인튜닝된 모델 테스트] ---

--- [테스트 1] ---
질문: 휴먼금융소프트는 언제 설립되었나요?
답변: 휴먼금융소프트는 2019년에 설립되었습니다.

--- [테스트 2] ---
질문: 휴먼금융소프트의 주요 제품은 무엇인가요?
답변: 휴먼금융소프트는 AURA (Advanced Unified Reasoning Assistant), Future-Edu (퓨처에듀), TeamSync (팀싱크) 등 시대를 앞서가는 AI 솔루션을 제공합니다.

--- [테스트 3] ---
질문: 휴먼금융소프트에 대해 알려주세요.
답변: 저희 휴먼금융소프트는 '미래의 소프트웨어, 오늘을 살다'는 비전 아래, 가장 진보된 AI 기술을 연구하고 상용화하고 있습니다.


In [ ]:
# 기타: 어댑터 다운로드 (로컬PC에서 파인튜닝된 모델활용 목적)
from google.colab import files
import shutil
import os

print("📦 어댑터 폴더 압축 중...")

# 어댑터 폴더를 ZIP으로 압축
adapter_folder = "gemma2-lora-adapters-kor"  # 학습 후 저장된 어댑터 폴더

if os.path.exists(adapter_folder):
    print(f"✅ 폴더 발견: {adapter_folder}")

    # ZIP 파일 생성
    shutil.make_archive('gemma2-lora-adapters-kor', 'zip', '.', adapter_folder)
    print("✅ ZIP 파일 생성 완료!")

    # 파일 크기 확인
    zip_size = os.path.getsize('gemma2-lora-adapters-kor.zip') / (1024**2)
    print(f"📊 파일 크기: {zip_size:.2f} MB")

    # 다운로드
    print("⬇️ 다운로드 시작...")
    files.download('gemma2-lora-adapters-kor.zip')
    print("✅ 다운로드 완료!")
else:
    print(f"❌ 폴더를 찾을 수 없습니다: {adapter_folder}")
    print("   학습이 완료되었는지 확인하세요.")

📦 어댑터 폴더 압축 중...
✅ 폴더 발견: gemma2-lora-adapters-kor
✅ ZIP 파일 생성 완료!
📊 파일 크기: 80.43 MB
⬇️ 다운로드 시작...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ 다운로드 완료!
